# Notebook 38 — Prelim R5 Final Sprint

Runs exactly `TRUE_BCF1`, `SAFE_R4_LIVE_WINNER`, `SAFE_R5_QE`, and `SAFE_R5_GATED` on separate `DEV_CROSS_60` and `DEV_L21_150` benchmarks. All prediction files are validated and SHA-256 frozen before the runner reads either GT file. This notebook never opens or searches `SEALED_FINAL_30`, never runs Whisper, never rebuilds corpus embeddings or the SigLIP2 index, never downloads a model, and never uploads a submission.

The runner uses five deterministic query views, existing A0/S1 indices, external ASR lexical/E5 evidence, the existing XCLIP/EventGraph tail, and deterministic evidence-first QA with exact BCF1 fallback. The final downloadable artifact is `/kaggle/working/prelim_r5_final_sprint_bundle.zip`.

In [ ]:
import os
from pathlib import Path

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path(os.environ.get("AIC_REPO_DIR", "/kaggle/working/AIC2026_TeamPTK_SGU"))
REFRESH_REPO = os.environ.get("AIC_REFRESH_REPO", "1") == "1"
ANCHOR = "d338d8d809bbb9e057e18becc607af2eb3bea254"
INPUTS = {
    "raw_dataset": os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"),
    "team_eval_cross60_l21": os.environ.get(
        "AIC_TEAM_EVAL_DEV_ROOT", "/kaggle/input/datasets/irthn1311/aic2026_team_eval_dev_v1"
    ),
    "fs1_master_freeze_bcf1": os.environ.get(
        "AIC_FS1_MASTER_FREEZE_ROOT",
        "/kaggle/input/datasets/irthn1311/fs1-master-preparation-freeze-2026-08-18",
    ),
    "stage1_exact_index": os.environ.get(
        "AIC_STAGE1_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle"
    ),
    "stage1b_verified_contract": os.environ.get(
        "AIC_STAGE1B_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports",
    ),
    "stage1e_language_contract": os.environ.get(
        "AIC_STAGE1E_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze",
    ),
    "openai_clip_offline_asset": os.environ.get(
        "AIC_CLIP_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32"
    ),
    "opus_mt_vi_en_offline_asset": os.environ.get(
        "AIC_OPUS_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en"
    ),
    "siglip2_offline_asset": os.environ.get(
        "AIC_SIGLIP2_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-siglip2-base-patch16-224"
    ),
    "exact_prebuilt_siglip2_index": os.environ.get(
        "AIC_SCA1_INDEX_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-sca1-siglip2-index-v01"
    ),
    "completion_xclip_dino_ocr_evidence": os.environ.get(
        "AIC_COMPLETION_EVIDENCE_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-completion-v11-evidence-bundle",
    ),
    "external_asr_v3_validated": os.environ.get(
        "AIC_ASR_EXTERNAL_V3_ROOT",
        "/kaggle/input/datasets/irthn1311/asr-external-v3-validated-bundle",
    ),
    "multilingual_e5_onnx_query_encoder": os.environ.get(
        "AIC_E5_ROOT",
        "/kaggle/input/datasets/irthn1311/aic2026-multilingual-e5-small-onnx-query-encoder",
    ),
}
OUTPUT_ROOT = Path("/kaggle/working/prelim_r5_final_sprint")
OUTPUT_ZIP = Path("/kaggle/working/prelim_r5_final_sprint_bundle.zip")
print(
    {
        "required_inputs": INPUTS,
        "internet_required": "GIT_CLONE_OR_REFRESH; PYTHON_DEPENDENCY_INSTALL_ONLY_IF_MISSING",
        "model_download_required": False,
        "whisper_run": False,
        "corpus_preprocessing": False,
        "siglip2_index_rebuild": False,
        "qwen_enabled": False,
        "output_zip": str(OUTPUT_ZIP),
    }
)

In [ ]:
import subprocess
import sys


def git_result(*args):
    return subprocess.run(
        ["git", *args],
        cwd=REPO_DIR if (REPO_DIR / ".git").is_dir() else None,
        capture_output=True,
        text=True,
        check=False,
    )


def git(*args):
    result = git_result(*args)
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout.strip()


if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout")
if not (REPO_DIR / ".git").is_dir():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True
    )
target = None
if not REFRESH_REPO:
    for candidate in (REPO_REF, f"origin/{REPO_REF}"):
        probe = git_result("rev-parse", "--verify", f"{candidate}^{{commit}}")
        if probe.returncode == 0:
            target = probe.stdout.strip()
            break
if target is None:
    git("fetch", "--no-tags", "origin", REPO_REF)
    target = "FETCH_HEAD"
git("checkout", "--detach", target)
HEAD = git("rev-parse", "HEAD")
if git_result("cat-file", "-e", f"{ANCHOR}^{{commit}}").returncode != 0:
    git("fetch", "--no-tags", "origin", REPO_REF)
ancestor = git_result("merge-base", "--is-ancestor", ANCHOR, HEAD).returncode == 0
if not ancestor:
    raise RuntimeError(f"R5 lineage gate failed: anchor={ANCHOR}, HEAD={HEAD}")
required = [
    REPO_DIR / "src/triage_eg/prelim_r5/runner.py",
    REPO_DIR / "scripts/run_prelim_r5_final.py",
    REPO_DIR / "configs/experiments/triage_prelim_r5_final.yaml",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise RuntimeError(f"R5 source missing from resolved ref: {missing}")
sys.path.insert(0, str(REPO_DIR / "src"))
print(
    {
        "source_ref": REPO_REF,
        "HEAD": HEAD,
        "anchor_is_ancestor": ancestor,
        "checkout_mode": "DETACHED_FETCHED_REF",
        "git_status": git("status", "--short") or "CLEAN",
    }
)

In [ ]:
import importlib.util
import re

missing_packages = []
if importlib.util.find_spec("onnxruntime") is None:
    missing_packages.append("onnxruntime==1.20.1")
if importlib.util.find_spec("tokenizers") is None:
    missing_packages.append("tokenizers==0.21.0")
if missing_packages:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--disable-pip-version-check",
            *missing_packages,
        ],
        check=True,
    )
test_env = dict(os.environ)
test_env["PYTHONPATH"] = str(REPO_DIR / "src") + (
    os.pathsep + test_env["PYTHONPATH"] if test_env.get("PYTHONPATH") else ""
)
command = [
    sys.executable,
    "-m",
    "pytest",
    "tests/unit/prelim_r5",
    "tests/unit/fs1_v11",
    "tests/unit/bcf1_protected_late_fusion",
    "tests/unit/sca1_siglip2_complementarity",
    "-q",
]
test = subprocess.run(
    command, cwd=REPO_DIR, env=test_env, capture_output=True, text=True, check=False
)
text = (test.stdout + "\n" + test.stderr).strip()
summary = {
    "returncode": test.returncode,
    "passed": int(re.search(r"(\d+) passed", text).group(1))
    if re.search(r"(\d+) passed", text)
    else None,
    "output_tail": text.splitlines()[-30:],
}
print({"dependency_install": missing_packages or "NOT_REQUIRED", "tests": summary})
if test.returncode:
    raise RuntimeError("R5 required tests failed before prediction")

In [ ]:
run_env = dict(test_env)
run_env.update(
    {
        "AIC_REPO_DIR": str(REPO_DIR),
        "AIC_R5_OUTPUT_ROOT": str(OUTPUT_ROOT),
        "AIC_R5_OUTPUT_ZIP": str(OUTPUT_ZIP),
    }
)
run = subprocess.run(
    [sys.executable, str(REPO_DIR / "scripts/run_prelim_r5_final.py")],
    cwd=REPO_DIR,
    env=run_env,
    text=True,
    check=False,
)
if run.returncode:
    raise RuntimeError(f"R5 runner failed with exit code {run.returncode}")

In [ ]:
import hashlib
import json
import zipfile

if not OUTPUT_ZIP.is_file():
    raise RuntimeError(f"R5 output ZIP missing: {OUTPUT_ZIP}")
digest = hashlib.sha256()
with OUTPUT_ZIP.open("rb") as stream:
    for block in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(block)
required_members = {
    "R5_FINAL_DECISION.md",
    "pre_gt_prediction_hashes.json",
    "cross60_scores.json",
    "l21_scores.json",
    "r5_query_view_diagnostics.jsonl",
    "r5_candidate_comparison.csv",
    "r5_candidate_comparison.json",
    "r5_head_override_audit.jsonl",
    "r5_qa_deterministic_evidence.jsonl",
    "production_policy.json",
    "run_provenance.json",
}
with zipfile.ZipFile(OUTPUT_ZIP) as archive:
    members = set(archive.namelist())
    missing = sorted(required_members - members)
    forbidden = sorted(
        name for name in members if "sealed" in name.casefold() or name.endswith("gt.jsonl")
    )
    if missing or forbidden:
        raise RuntimeError({"missing_members": missing, "forbidden_members": forbidden})
policy = json.loads((OUTPUT_ROOT / "production_policy.json").read_text(encoding="utf-8"))
print(
    {
        "DOWNLOAD_ZIP": str(OUTPUT_ZIP),
        "size_bytes": OUTPUT_ZIP.stat().st_size,
        "sha256": digest.hexdigest(),
        "members": len(members),
        "production_recommendation": policy["production_policy"],
        "submission_uploaded": False,
    }
)